# Detector convergence pilot for the full-data balancing experiment

`real_only` on the full train split (217 ripe, 84 unripe; DSC_0093 excluded),
validation split only. Four runs, two per GPU on T4 x2:

    seed 0: 3000 updates, 4500 updates      seed 1: 3000 updates, 4500 updates

Each run has its own cosine schedule and scores validation every 250 updates. The
pre-registered rule in `Ujicoba CycleGAN/BALANCING_PLAN.md` decides from these runs
whether 3000 updates are past convergence. The budget is not chosen per arm, and
the test split is never opened.

In [ ]:
# The input mounts are read only and ultralytics writes next to its working
# directory, so the whole code tree is copied into /kaggle/working first.
import os, shutil, subprocess, sys, time
from pathlib import Path

INPUT = Path('/kaggle/input')
print('mounted:', sorted(p.name for p in INPUT.iterdir()) if INPUT.exists() else 'nothing')

def find(*names, marker=None):
    # Kaggle mounts a dataset at /kaggle/input/<slug> or, when several are
    # attached, at /kaggle/input/datasets/<owner>/<slug>, so the slug is searched
    # by name at any depth before falling back to the marker file.
    for name in names:
        for candidate in [INPUT / name, *sorted(INPUT.rglob(name))]:
            if candidate.is_dir() and (marker is None or (candidate / marker).exists()):
                return candidate
    hits = sorted({p.parent for p in INPUT.rglob(marker)}) if marker else []
    if len(hits) == 1: return hits[0]
    raise SystemExit(f'cannot locate {names} (marker {marker}); found {hits}')

SOURCE = find('melon-cyclegan-code', marker='Ujicoba CycleGAN')
DATASET = find('melon-dataset-regrouped-v4', 'melon-final-reviewed-v4', marker='manifest.csv')
ASSETS = find('melon-benchmark-assets', marker='reviewed_masks.json')

WORK = Path('/kaggle/working')
CODE = WORK / 'code'
if CODE.exists(): shutil.rmtree(CODE)
shutil.copytree(SOURCE, CODE)
HERE = CODE / 'Ujicoba CycleGAN'
os.environ['MELON_DATASET'] = str(DATASET)
os.environ['MELON_ASSETS'] = str(ASSETS)
sys.path.insert(0, str(HERE))

import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('code:', sorted(p.name for p in HERE.iterdir()))
print('dataset:', (DATASET / 'manifest.csv').exists(), 'assets:', (ASSETS / 'reviewed_masks.json').exists())


def run(*command, cwd=None):
    started = time.time()
    result = subprocess.run([sys.executable, *[str(c) for c in command]], cwd=str(cwd or HERE))
    print(f'-> returncode {result.returncode} in {time.time() - started:.1f}s', flush=True)
    if result.returncode != 0: raise SystemExit(result.returncode)

In [ ]:
# The Kaggle image no longer ships ultralytics; the detector needs it. Nothing
# else is installed, so torch and torchvision stay at the image versions.
import importlib.util
if importlib.util.find_spec('ultralytics') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
import importlib.metadata as meta
for package in ('torch', 'torchvision', 'ultralytics'):
    try: print(package, meta.version(package))
    except meta.PackageNotFoundError: print(package, 'NOT INSTALLED')

In [ ]:
import json
ARMS = WORK / 'balance_arms_pilot'
run(HERE / 'tools' / 'build_balance_arms.py', '--real-only', '--out', ARMS, '--dataset', DATASET)
print(json.loads((ARMS / 'arms_summary.json').read_text()))

In [ ]:
import torch, subprocess
TRAINER = HERE.parent / '14_Benchmark_Kaggle' / 'tools' / 'train_detector.py'
PLAN = {0: [(0, 3000), (0, 4500)], 1: [(1, 3000), (1, 4500)]}
n_gpu = torch.cuda.device_count()
print('visible GPUs:', n_gpu)
def chain(gpu, runs):
    lines = [f"CUDA_VISIBLE_DEVICES={gpu if n_gpu >= 2 else 0} {sys.executable} {TRAINER} --arm {ARMS / 'real_only'} "
             f"--seed {seed} --updates {updates} --batch 8 --lr 1e-4 --eval-every 250 "
             f"--out {WORK / f'pilot_u{updates}'}" for seed, updates in runs]
    log = open(WORK / f'pilot_gpu{gpu}.log', 'w')
    return subprocess.Popen(' && '.join(lines), shell=True, cwd=str(TRAINER.parent.parent),
                            stdout=log, stderr=subprocess.STDOUT)
started = time.time()
if n_gpu >= 2:
    codes = [p.wait() for p in [chain(g, r) for g, r in PLAN.items()]]
else:
    codes = [chain(g, r).wait() for g, r in PLAN.items()]
print('return codes', codes, 'wall minutes', round((time.time() - started) / 60, 1))
for g in PLAN: print((WORK / f'pilot_gpu{g}.log').read_text()[-1500:])
if any(codes): raise SystemExit('pilot training failed')

In [ ]:
import statistics
rows = {}
for updates in (3000, 4500):
    for seed in (0, 1):
        run_dir = WORK / f'pilot_u{updates}' / f'real_only__seed{seed}'
        summary = json.loads((run_dir / 'run_summary.json').read_text())
        history = json.loads((run_dir / 'valid_history.json').read_text())
        rows[f'{updates}@{seed}'] = dict(final=summary['validation']['map50_95'],
                                         selected=summary['selected']['validation']['map50_95'],
                                         selected_update=summary['selected']['update'],
                                         curve=[(h['update'], round(h['map50_95'], 4)) for h in history],
                                         seconds=summary['training_seconds'])
mean = lambda u, k: statistics.fmean(rows[f'{u}@{s}'][k] for s in (0, 1))
report = dict(runs=rows, mean_final_3000=mean(3000, 'final'), mean_final_4500=mean(4500, 'final'),
              delta_final_4500_minus_3000=mean(4500, 'final') - mean(3000, 'final'),
              rule='3000 updates are sufficient if mean final validation mAP50:95 at 4500 minus at 3000 <= 0.010',
              test_used=False)
report['sufficient'] = report['delta_final_4500_minus_3000'] <= 0.010
(WORK / 'pilot_convergence.json').write_text(json.dumps(report, indent=2) + '\n')
print(json.dumps({k: v for k, v in report.items() if k != 'runs'}, indent=2))
for key, value in rows.items(): print(key, value['final'], value['selected'], value['selected_update'], value['curve'])

In [ ]:
shutil.rmtree(CODE, ignore_errors=True)
shutil.rmtree(ARMS, ignore_errors=True)
for path in WORK.glob('pilot_u*/*/weights*.pt'): path.unlink()
print(sorted(p.name for p in WORK.iterdir()))